# 02_人資_教育訓練名單

文字條件篩選與計數

任務：列出員工編號與完成狀態。；篩出未完成人員，保留員工編號、姓名與部門。；計算完成人數與完成率。

規則：每位員工只有一列，完成率＝完成人數 ÷ 全部員工人數；篩選文字要完全相同。

先人工核對再執行。資料皆為教學模擬值。

In [ ]:
%pip -q install openpyxl numpy matplotlib

In [ ]:
from pathlib import Path
import csv
from openpyxl import load_workbook

# 本機：把 Notebook 與 Excel 放在同一資料夾。Colab：先上傳本例 Excel。
SOURCE = Path('02_人資_教育訓練名單.xlsx')
if not SOURCE.exists():
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(f'請將 {SOURCE.name} 放到目前工作目錄：{Path.cwd()}')
    files.upload()
if not SOURCE.exists():
    raise FileNotFoundError(f'尚未找到 {SOURCE.name}，請確認上傳檔名')
OUT = Path('output') / '02_人資_教育訓練名單'
OUT.mkdir(parents=True, exist_ok=True)

def read_rows(sheet):
    data = sheet.iter_rows(values_only=True)
    headers = next(data)
    return [dict(zip(headers, row)) for row in data if any(v is not None for v in row)]

def save_csv(name, records):
    if not records:
        print('沒有符合條件的資料，未輸出 CSV')
        return
    with (OUT / name).open('w', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=list(records[0]))
        writer.writeheader()
        writer.writerows(records)

# 來源檔只有原始值，沒有需要重算的 Excel 公式。
wb = load_workbook(SOURCE, read_only=True, data_only=True)
print('工作表：', wb.sheetnames)
rows = read_rows(wb['資料'])
print('第一筆：', rows[0])

In [ ]:
pending = [r for r in rows if r['完成狀態'] == '未完成']
completed = sum(r['完成狀態'] == '完成' for r in rows)
print('待完成：', pending)
print('完成率：', f'{completed / len(rows):.1%}')
assert completed == 4 and len(pending) == 2
wb.close()

## 驗收答案

完成 4 人、未完成 2 人；完成率 66.7%，待完成為 E002、E004。